# Lab 1: Poisson's Equation and Planetary Gravity

| | |
|---|---|
| **Module** | M1 — Potential Fields and Boundary Value Problems |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M1.1–M1.3; Homework 1 Problems 1.3–1.4 |
| **Textbook** | Blakely Ch. 1–3; lecture notes M1 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Implement** a finite-difference solver for the spherically-symmetric Poisson equation using a change of variables.
2. **Compare** the numerical and analytical gravitational potentials for a uniform sphere.
3. **Compute** the gravitational acceleration field inside and outside a sphere from the potential.
4. **Apply** the uniform-density model to predict surface gravity for Earth and Jupiter.

---

## How to use this notebook

Cells are tagged in their first line:

- **`[PROVIDED]`** — run as-is, do not modify
- **`[IMPLEMENT]`** — replace each `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check your work; prints ✓ PASS or ✗ FAIL with diagnostics
- **`[EXPLORE]`** — starting point; modify freely to answer the question above it

Markdown cells with **Your response:** are written-answer questions — write directly  
in the notebook. These are graded.

Need a hint? Run `print(hints['key'])` in a code cell.  
Hint keys are listed at the start of each Part.

---

> **A note on AI tools:** You are welcome to use AI assistants in this lab.
> However, **all written answers must be grounded in specific results from your own notebook run** —
> numerical values from your VALIDATE cells, features you observe in your plots, or choices you made in your code.
> Answers that could have been written without running the notebook will not receive credit for the written component,
> regardless of their conceptual accuracy.

# Lab 1 - Gravitational Fields of Uniform Planets

_You are welcome to use AI tools. All written answers must reference specific numerical results from your notebook — plot features, VALIDATE output, numbers from your calculations. Answers that could have been written without running the notebook will not receive credit for the written component._

## Background

The gravitational potential $U(\mathbf{r})$ of a body with density distribution $\rho(\mathbf{r})$
obeys **Poisson’s equation**:

$$
\nabla^2 U = -4\pi G \rho
$$

where $G = 6.674 \times 10^{-11} \text{N } \text{m}^2 \text{kg}^{-2}$ is Newton’s constant.
In mass-free regions ($\rho = 0$) this reduces to **Laplace’s equation** $\nabla^2 U = 0$.
We follow the Blakely (Ch. 3) convention where $U > 0$ and the gravitational
acceleration is $\mathbf{g} = -\nabla U$.

For a **uniform sphere** of radius $R$, density $\rho_0$, and total mass
$M = \frac{4}{3}\pi \rho_0 R^3$, spherical symmetry turns Poisson’s equation
into a 1-D ODE with the closed-form solution:

$$
U(r) = \begin{cases}
\dfrac{2}{3}\pi G \rho_0 (3R^2 - r^2) & r \leq R \\[10pt]
\dfrac{GM}{r} & r > R
\end{cases}
$$

The gravitational acceleration magnitude $g(r) = -dU/dr$ (radially inward, $g > 0$) is:

$$
g(r) = \begin{cases}
\dfrac{4}{3}\pi G \rho_0\, r & r \leq R \\[10pt]
\dfrac{GM}{r^2} & r > R
\end{cases}
$$

$g$ increases **linearly** from zero at the center, peaks at the surface
($g_\text{surface} = GM/R^2 = \frac{4}{3}\pi G\rho_0 R$), then decays as $1/r^2$ outside.
In this lab you will derive these results numerically by solving Poisson’s equation
on a grid, then apply the model to predict surface gravity for Earth and Jupiter.

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────────────────
# Imports and constants. Run this cell first.

import numpy as np
import matplotlib.pyplot as plt

# Physical constant
G = 6.674e-11    # gravitational constant, N m^2 kg^-2

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────────────────
# Hint dictionary.  Access with:  print(hints['key'])
# Part 1 keys: 'p1_density', 'p1_inside', 'p1_substitution', 'p1_bc', 'p1_recovery'
# Part 2 keys: 'p2_gravity', 'p2_surface', 'p2_numerical'

hints = {
    # Part 1
    'p1_density': (
        'The sphere has density rho0 for r <= R and zero outside. '
        'np.where(condition, val_if_true, val_if_false) handles arrays cleanly.'
    ),
    'p1_inside': (
        'Inside (r <= R):  U(r) = (2/3)*pi*G*rho0*(3*R**2 - r**2). '
        'Outside (r > R):  U(r) = G*M/r,  M = (4/3)*pi*rho0*R**3. '
        'Verify that both give the same value at r = R.'
    ),
    'p1_substitution': (
        'Let u(r) = r*U(r). The spherical-symmetry Poisson equation '
        '(1/r^2) d/dr[r^2 dU/dr] = -4*pi*G*rho '
        'becomes d^2u/dr^2 = -4*pi*G*rho(r)*r. '
        'Discretise with the centred stencil: '
        '(u[i-1] - 2*u[i] + u[i+1]) / dr**2 = -4*pi*G*rho[i]*r[i]. '
        'The N-2 interior equations (i=1,...,N-2) form a tridiagonal linear system.'
    ),
    'p1_bc': (
        'Two BCs pin the solution uniquely. '
        '(1) u(0) = 0: U must be finite at r=0, so u = r*U -> 0. '
        '(2) u(r_max) = G*M: U(r_max) = GM/r_max gives u = r*U = G*M. '
        'Incorporate each BC by subtracting (bc_value / dr**2) from the '
        'corresponding end of the RHS vector f.'
    ),
    'p1_recovery': (
        'After solving for u_interior, assemble '
        'u_full = [0, u_interior, G*M]. '
        'Then U[i] = u_full[i] / r[i] for i >= 1. '
        'At i=0 (r=0) use L\'Hopital: U[0] = lim u/r = du/dr|_{r=0} ~= u_full[1] / dr.'
    ),
    # Part 2
    'p2_gravity': (
        'Inside:  g(r) = (4/3)*pi*G*rho0*r  (linear, zero at center). '
        'Outside: g(r) = G*M / r**2         (1/r^2 decay). '
        'g is the magnitude of the radially-inward acceleration; '
        'it equals -dU/dr since U decreases outward.'
    ),
    'p2_surface': (
        'Surface gravity: g_surface = (4/3)*pi*G*rho0*R = G*M/R**2. '
        'This depends on BOTH density AND radius. '
        'A less dense but much larger planet can still have strong surface gravity.'
    ),
    'p2_numerical': (
        'Compute g numerically with -np.gradient(U_fd, r). '
        'Since dU/dr < 0 (potential decreases outward), the minus sign gives g > 0. '
        'Expect <1% agreement with the analytical formula away from r=0 and r=R.'
    ),
}

print("Hints loaded.  Access with: print(hints['key'])")

---
## Part 1: Solving Poisson’s Equation Numerically *(Guided — ~40 min)*

Inside the sphere, Poisson’s equation holds; outside, it reduces to Laplace’s equation:

| Region | Equation | Name |
|--------|----------|---------|
| $r \leq R$ | $\nabla^2 U = -4\pi G \rho_0$ | Poisson |
| $r > R$    | $\nabla^2 U = 0$              | Laplace |

By setting $\rho = 0$ outside the sphere and solving on a single domain, you handle
both equations simultaneously. The key trick is the **change of variables** $u(r) = r\,U(r)$,
which converts the spherically-symmetric Laplacian into a standard 1-D second derivative:

$$
\frac{1}{r^2}\frac{d}{dr}\!\left(r^2\frac{dU}{dr}\right) = -4\pi G\rho
\quad\Longrightarrow\quad
\frac{d^2 u}{dr^2} = -4\pi G\,\rho(r)\cdot r
$$

This is a textbook tridiagonal system — fast to assemble and solve.

**Available hints:** `p1_density`, `p1_inside`, `p1_substitution`, `p1_bc`, `p1_recovery`

In [ ]:
# [IMPLEMENT] ──────────────────────────────────────────────────────────────────────────────
# Tier 1: The source term in Poisson's equation.

def density_profile(r, rho0, R):
    """
    Density of a uniform sphere at radial coordinate r.

    Parameters
    ----------
    r : float or ndarray
        Radial distance from sphere center [m]. Must be >= 0.
    rho0 : float
        Constant interior density [kg/m^3].
    R : float
        Sphere radius [m].

    Returns
    -------
    rho : float or ndarray
        Density [kg/m^3]: rho0 for r <= R, zero for r > R.
    """
    r = np.asarray(r, dtype=float)
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────────────────
# Run this cell to check density_profile. Do not modify.

def _check(label, got, expected, rtol=1e-5, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  \u2713 PASS  {label}')
    else:
        print(f'  \u2717 FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

_R   = 6.371e6   # Earth-like radius, m
_rho = 5515.0    # Earth-like mean density, kg/m^3

_check('rho at center     (r = 0)',       density_profile(0.0,        _rho, _R), _rho)
_check('rho at surface    (r = R)',       density_profile(_R,         _rho, _R), _rho)
_check('rho just inside   (r = 0.999R)', density_profile(0.999*_R,   _rho, _R), _rho)
_check('rho outside       (r = 2R)',     density_profile(2.0*_R,     _rho, _R), 0.0)
_check('array: inside, surface, outside',
       density_profile(np.array([0.5*_R, _R, 1.5*_R]), _rho, _R),
       np.array([_rho, _rho, 0.0]))

In [ ]:
# [IMPLEMENT] ──────────────────────────────────────────────────────────────────────────────
# Tier 1: Analytical solution for U(r) -- reference for verifying the FD solver.

def analytical_potential(r, rho0, R):
    """
    Analytical gravitational potential of a uniform sphere.

    Blakely convention: U > 0, gravitational acceleration g = grad(U).

    Inside  (r <= R): U(r) = (2/3) * pi * G * rho0 * (3*R**2 - r**2)
    Outside (r >  R): U(r) = G * M / r,   M = (4/3) * pi * rho0 * R**3

    Parameters
    ----------
    r : float or ndarray
        Radial distance [m]. Must be >= 0 (use r > 0 for exterior formula).
    rho0 : float
        Uniform density [kg/m^3].
    R : float
        Sphere radius [m].

    Returns
    -------
    U : float or ndarray
        Gravitational potential [J/kg].

    Notes
    -----
    At r = R both expressions give U = GM/R (continuity). Verify this.
    At r = 0: U(0) = (3/2) * U(R)  -- the center value is 1.5x the surface value.
    """
    r = np.asarray(r, dtype=float)
    U = np.empty_like(r)
    M = (4.0 / 3.0) * np.pi * rho0 * R**3
    inside = r <= R

    # Step 1: potential inside the sphere
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    # Step 2: potential outside the sphere
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    return U

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────────────────
# Run this cell to check analytical_potential. Do not modify.

_M         = (4.0/3.0) * np.pi * _rho * _R**3
_GM        = G * _M
_U_surface = _GM / _R                   # approx 6.25e7 J/kg

_check('U at surface (r = R)',
       analytical_potential(_R, _rho, _R), _U_surface)
_check('U at center = (3/2) U_surface  (r = 0)',
       analytical_potential(0.0, _rho, _R), 1.5 * _U_surface)
_check('U at r = R/2 (inside)',
       analytical_potential(0.5*_R, _rho, _R),
       (2.0/3.0)*np.pi*G*_rho*(3.0*_R**2 - (0.5*_R)**2))
_check('U at r = 5R (far exterior)',
       analytical_potential(5.0*_R, _rho, _R), _GM / (5.0*_R))
_check('Continuity at surface: inside value = outside value',
       analytical_potential(_R*(1.0 - 1e-9), _rho, _R),
       analytical_potential(_R*(1.0 + 1e-9), _rho, _R), rtol=1e-5)

In [ ]:
# [IMPLEMENT] ──────────────────────────────────────────────────────────────────────────────
# Tier 1: Finite-difference solver for Poisson's equation.
#
# Change of variables:  u(r) = r * U(r)
#
# Transforms the spherical Poisson equation
#
#   (1/r^2) d/dr[ r^2 dU/dr ] = -4*pi*G*rho(r)
#
# into the standard 1-D form:
#
#   d^2u/dr^2  =  -4*pi*G * rho(r) * r
#
# Centred finite-difference stencil (second-order):
#
#   (u[i-1] - 2*u[i] + u[i+1]) / dr**2  =  -4*pi*G * rho[i] * r[i]
#
# Grid:    r[0] = 0,  r[N-1] = r_max,  spacing dr (uniform)
# Unknowns: u[1], u[2], ..., u[N-2]   (N-2 interior points)
# Boundary conditions:
#   u[0]   = 0      (regularity: U finite at r=0  =>  u = r*U -> 0)
#   u[N-1] = G*M    (Dirichlet:  U(r_max) = GM/r_max  =>  u = G*M)

def solve_poisson_fd(r, rho0, R):
    """
    Solve nabla^2 U = -4*pi*G*rho for a uniform sphere using finite differences.

    Parameters
    ----------
    r : ndarray, shape (N,)
        Radial grid [m]. Must start at 0 with uniform spacing dr.
    rho0 : float
        Uniform sphere density [kg/m^3].
    R : float
        Sphere radius [m].

    Returns
    -------
    U : ndarray, shape (N,)
        Gravitational potential at each grid point [J/kg].
    """
    N   = len(r)
    dr  = r[1] - r[0]
    M   = (4.0 / 3.0) * np.pi * rho0 * R**3

    # Known boundary values of u
    u_left  = 0.0      # u(r=0)     = 0
    u_right = G * M    # u(r=r_max) = G*M

    rho = density_profile(r, rho0, R)

    # ── Step 1: Build the RHS vector f ─────────────────────────────────────────
    # f has N-2 elements, one per interior point i = 1, ..., N-2.
    # f[k] = -4*pi*G * rho(r[k+1]) * r[k+1]   for k = 0, ..., N-3
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    # ── Step 2: Build the (N-2) x (N-2) tridiagonal matrix A ───────────────────
    # Main diagonal:   -2 / dr**2
    # Off-diagonals:    1 / dr**2
    # Hint: np.diag(values) builds a diagonal matrix;
    #       np.diag(values, k=1) and np.diag(values, k=-1) build super/sub-diagonals.
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    # ── Step 3: Apply boundary conditions to f ─────────────────────────────────
    # Left  BC (u[0]   = u_left):  f[0]  -= u_left  / dr**2
    # Right BC (u[N-1] = u_right): f[-1] -= u_right / dr**2
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    # ── Step 4: Solve the linear system  A @ u_interior = f ────────────────────
    # Hint: use np.linalg.solve(A, f)
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    # ── Step 5: Assemble u_full and recover U ────────────────────────────────
    # u_full = [u_left, u_interior (from Step 4), u_right]
    # U[i]   = u_full[i] / r[i]  for i >= 1
    # U[0]   = u_full[1] / dr   (L'Hopital limit: lim_{r->0} u/r = du/dr|_0)
    # REPLACE WHAT IS BELOW WITH YOUR CODE HERE
    raise NotImplementedError

    return U

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────────────────
# Compare the FD solver to the analytical solution on a moderately refined grid.
# Do not modify.

_N     = 600
_r_max = 3.0 * _R                       # extend to 3R -- well outside the sphere
_r_fd  = np.linspace(0.0, _r_max, _N)

_U_fd   = solve_poisson_fd(_r_fd, _rho, _R)
_U_anal = analytical_potential(_r_fd[1:], _rho, _R)   # skip r=0

_rel_err = np.abs(_U_fd[1:] - _U_anal) / np.abs(_U_anal)
_in      = _r_fd[1:] <= _R
_out     = _r_fd[1:] >  _R

_check('Max relative error inside  (expect < 5e-4)',
       _rel_err[_in].max(),  0.0, atol=5e-4)
_check('Max relative error outside (expect < 5e-4)',
       _rel_err[_out].max(), 0.0, atol=5e-4)
_check('U at surface (FD vs. analytical)',
       _U_fd[np.argmin(np.abs(_r_fd - _R))], _U_surface, rtol=5e-3)
_check('U at center  (FD) = (3/2) U_surface',
       _U_fd[0], 1.5 * _U_surface, rtol=5e-3)

In [ ]:
# [EXPLORE] ──────────────────────────────────────────────────────────────────────────────
# Plot the gravitational potential: FD solution vs. analytical.
# Extend this cell to annotate the sphere surface and the ratio U(0)/U(R).

fig, ax = plt.subplots(figsize=(8, 4))

r_plot = _r_fd[1:]   # skip r=0 for a cleaner plot
ax.plot(r_plot / _R, _U_fd[1:]  / 1e7, lw=2,        label='Finite differences')
ax.plot(r_plot / _R, _U_anal    / 1e7, lw=1.5, ls='--', label='Analytical')
ax.axvline(1.0, color='k', lw=0.8, ls=':', label='Sphere surface (r = R)')

ax.set_xlabel('r / R  (normalized radius)')
ax.set_ylabel('U(r)  [1e7 J/kg]')
ax.set_title('Gravitational potential of a uniform sphere')
ax.legend()
plt.tight_layout()
plt.show()

### Question 1.1

The potential $U(r)$ is governed by different equations inside and outside the sphere,
yet it is **continuous** at $r = R$.

(a) Your VALIDATE cell prints values of $U(0)$ and $U(R)$ (the center and surface potential).
    Fill in: $U(0) \approx$ \_\_\_ J/kg and $U(R) \approx$ \_\_\_ J/kg.
    Explain physically why the potential at the center is non-zero and *higher* than at the surface —
    what does it mean energetically to sit at the center of a gravitating body?  
(b) The density $\rho$ has a **step discontinuity** at $r = R$, yet $U$ is continuous there.
    Look at your plot near $r = R$: does the *slope* of $U$ appear to change abruptly at $r = R$,
    or is it also smooth? What does your observation imply about which derivatives of $U$
    are and are not continuous across a density interface?  
(c) From your plot, estimate the ratio $U(0)/U(R)$ and compare it to the analytical
    prediction of $3/2$.

**Your response:**

> *(Write 4–6 sentences. Fill in the blanks in (a) before explaining. Replace this line.)*

### Question 1.2

The change of variables $u(r) = r \cdot U(r)$ was essential for the numerical approach.

(a) What boundary condition did you apply at $r = 0$ for $u$? Where does it come from
    physically?  
(b) What would happen numerically if you tried to solve directly for $U(r)$ near
    $r = 0$ without this substitution?

**Your response:**

> *(Write 4–6 sentences. Replace this line.)*

---
## Part 2: Surface Gravity of Earth and Jupiter *(Supported — ~50 min)*

You now have both a numerical ($U_\text{FD}$) and an analytical ($U_\text{anal}$)
expression for the potential. The gravitational acceleration magnitude is

$$
g(r) = -\frac{dU}{dr} =
\begin{cases}
\dfrac{4}{3}\pi G \rho_0\, r & r \leq R \\[10pt]
\dfrac{GM}{r^2}              & r > R
\end{cases}
$$

Your tasks:

1. Implement `analytical_gravity` from the docstring alone.
2. Apply it to Earth and Jupiter using their average densities and radii.
3. Compare to a numerical estimate of $g$ derived from the FD potential via `np.gradient`.

**Available hints:** `p2_gravity`, `p2_surface`, `p2_numerical`

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────────────────
# Planet parameters (IAU 2015 / NASA fact sheets). Run as-is.

planets = {
    'Earth': {
        'R':     6.371e6,   # mean radius [m]
        'rho':   5515.0,    # mean density [kg/m^3]
        'g_ref': 9.807,     # standard surface gravity [m/s^2]
    },
    'Jupiter': {
        'R':     6.9911e7,  # mean volumetric radius [m]
        'rho':   1326.0,    # mean density [kg/m^3]
        'g_ref': 24.79,     # mean surface gravity [m/s^2] (equatorial, rotation excluded)
    },
}

print('Planet parameters:')
for name, p in planets.items():
    print(f"  {name:8s}  R = {p['R']:.4e} m   rho = {p['rho']:6.0f} kg/m^3   g_ref = {p['g_ref']:.3f} m/s^2")

In [ ]:
# [IMPLEMENT] ──────────────────────────────────────────────────────────────────────────────
# Tier 2: Implement from the docstring alone. Use hints if needed.

def analytical_gravity(r, rho0, R):
    """
    Magnitude of gravitational acceleration for a uniform sphere.

    g(r) = (4/3)*pi*G*rho0*r   for r <= R  (linear from center)
    g(r) = G*M / r**2          for r >  R  (1/r^2 decay)

    g is positive and directed radially inward. Equivalently, g = -dU/dr
    (U decreases outward, so dU/dr < 0, and -dU/dr > 0).

    At r = R both expressions give g = GM/R**2 = (4/3)*pi*G*rho0*R.

    Parameters
    ----------
    r : float or ndarray
        Radial distance [m]. Must be >= 0.
    rho0 : float
        Uniform density [kg/m^3].
    R : float
        Sphere radius [m].

    Returns
    -------
    g : float or ndarray
        Gravitational acceleration magnitude [m/s^2].
    """
    r = np.asarray(r, dtype=float)
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────────────────
# Do not modify.

_g_surface = _GM / _R**2    # approx 9.82 m/s^2 for Earth parameters

_check('g at center = 0',
       analytical_gravity(0.0, _rho, _R), 0.0, atol=1e-10)
_check('g at r = R/2 = (1/2) g_surface  (linear inside)',
       analytical_gravity(0.5*_R, _rho, _R), 0.5 * _g_surface, rtol=1e-5)
_check('g continuous at surface: inside == outside',
       analytical_gravity(_R*(1 - 1e-9), _rho, _R),
       analytical_gravity(_R*(1 + 1e-9), _rho, _R), rtol=1e-5)
_check('g at r = 2R = (1/4) g_surface  (1/r^2 outside)',
       analytical_gravity(2.0*_R, _rho, _R), 0.25 * _g_surface, rtol=1e-5)

print()
print('Uniform-density model surface gravity:')
for name, p in planets.items():
    g_model = (4.0/3.0) * np.pi * G * p['rho'] * p['R']
    print(f"  {name:8s}  g_model = {g_model:.3f} m/s^2   g_ref = {p['g_ref']:.3f} m/s^2   "
          f"diff = {100*(g_model - p['g_ref'])/p['g_ref']:+.1f}%")

In [ ]:
# [EXPLORE] ──────────────────────────────────────────────────────────────────────────────
# Plot g(r) for Earth and Jupiter -- analytical and numerical (FD-based).
#
# For each planet:
#   1. Build a grid from 0 to 3R.
#   2. Compute g analytically with analytical_gravity.
#   3. Solve Poisson with solve_poisson_fd, then estimate g numerically:
#        g_num = -np.gradient(U_fd, r)
#   4. Plot both curves vs r/R. Mark the surface with a vertical line.
#
# Requirements:
#   - Two subplots side by side, one per planet.
#   - x-axis: r/R (normalised),  y-axis: g [m/s^2].
#   - Annotate the surface gravity value on each plot.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, p) in zip(axes, planets.items()):
    R_p   = p['R']
    rho_p = p['rho']
    N_p   = 600
    r_p   = np.linspace(0.0, 3.0*R_p, N_p)

    g_anal = analytical_gravity(r_p, rho_p, R_p)

    U_fd_p = solve_poisson_fd(r_p, rho_p, R_p)
    g_num  = -np.gradient(U_fd_p, r_p)

    ax.plot(r_p / R_p, g_anal, lw=2,        label='Analytical')
    ax.plot(r_p / R_p, g_num,  lw=1.5, ls='--', label='Numerical (FD)')
    ax.axvline(1.0, color='k', lw=0.8, ls=':', label='Surface')
    ax.set_xlabel('r / R')
    ax.set_ylabel('g  [m/s$^2$]')
    ax.set_title(name)
    ax.legend()

plt.suptitle('Gravitational acceleration — uniform density model', y=1.01)
plt.tight_layout()
plt.show()

### Question 2.1

Jupiter has a **lower** average density than Earth (1326 vs. 5515 kg/m$^3$),
yet your model predicts **stronger** surface gravity for Jupiter (~25.9 vs. ~9.8 m/s$^2$).

Using the formula $g_\text{surface} = \frac{4}{3}\pi G \rho_0 R$, explain quantitatively
why this is the case. Include a brief numerical comparison that identifies which factor
($\rho_0$ or $R$) dominates.

**Your response:**

> *(Write 3–5 sentences. Show a brief calculation. Replace this line.)*

### Question 2.2

(a) Your VALIDATE cell printed the uniform-density model surface gravity for Earth.
    Fill in: $g_\text{model} =$ \_\_\_ m/s². Compute the percentage error vs. the standard 9.807 m/s².
    Then name **two physical effects** the model ignores that could account for this discrepancy.
    For each, state whether it would *increase* or *decrease* $g$ relative to your model value,
    and explain why.  
(b) Look at your $g(r)$ plot for Earth: inside the sphere, the curve rises linearly from 0 to $g_\text{surface}$.
    Earth has a dense iron core (~11,000 kg/m$^3$) and a lighter crust (~2,800 kg/m$^3$).
    Describe in words how you would expect the true $g(r)$ curve to differ from the linear ramp you plotted:
    where would the slope be steeper, where flatter, and would the peak still occur exactly at $r = R$?

**Your response:**

> *(Write 4–6 sentences. Fill in the blank in (a) before explaining. Replace this line.)*

---
## Part 3: Convergence of the Finite-Difference Solver *(Open — ~40 min)*

A numerical solver is only reliable if you understand how its error scales with
grid resolution. Theory predicts that a second-order centred-difference scheme has
error $\propto (\Delta r)^2 \propto N^{-2}$: doubling the number of grid points
should reduce the error by a factor of four.

Your task is to verify this prediction empirically and identify any regions where
convergence is slower than expected.

**No hints provided for this part — design the investigation yourself.**

In [ ]:
# [EXPLORE] ──────────────────────────────────────────────────────────────────────────────
# Convergence study: how does FD error scale with grid resolution N?
#
# Suggested approach:
#   1. Choose a range of N values, e.g. [100, 200, 400, 800].
#      Note: np.linalg.solve is O(N^3); keep N <= 1000 for speed.
#   2. For each N, build a grid from 0 to 3*_R and call solve_poisson_fd.
#   3. Compare to analytical_potential at all interior points (skip r=0).
#      Record the maximum relative error: max |U_fd - U_anal| / |U_anal|.
#   4. Plot log10(max_error) vs. log10(N). Fit a line with np.polyfit to
#      estimate the convergence order (slope).
#
# Expected slope: -2  (second-order FD, error proportional to dr^2 = (r_max/(N-1))^2)

N_values   = np.array([100, 200, 400, 800])
max_errors = np.empty(len(N_values))

for k, N in enumerate(N_values):
    # YOUR CODE HERE
    pass

# --- log-log plot ---
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(N_values, max_errors, 'o-', lw=1.5)
ax.set_xlabel('N  (number of grid points)')
ax.set_ylabel('Max relative error')
ax.set_title('Convergence of the FD Poisson solver')

# Uncomment to estimate and annotate the slope:
# slope, intercept = np.polyfit(np.log10(N_values), np.log10(max_errors), 1)
# ax.loglog(N_values, 10**intercept * N_values**slope, 'k--',
#           label=f'slope = {slope:.2f}')
# ax.legend()
# print(f'Estimated convergence order: {slope:.2f}  (expect -2)')

plt.tight_layout()
plt.show()

### Question 3.1

Describe the convergence behavior you observed.

(a) Report the slope you measured in the log–log plot. Does it agree with the theoretical
    prediction of $-2$ for a second-order scheme?  
(b) Fill in from your convergence data at the finest $N$: max relative error
    *inside* the sphere $\approx$ \_\_\_, *outside* $\approx$ \_\_\_.
    Which region has the larger error? The density $\rho(r)$ has a step discontinuity at $r = R$.
    Explain why this particular location might reduce the local convergence rate.  
(c) For a practical application — say, computing surface gravity to within 0.1% —
    what minimum $N$ would your results suggest? Read this directly from your convergence plot.

**Your response:**

> *(Write 4–6 sentences. Fill in the blanks in (b) from your plot before explaining. Replace this line.)*

In [ ]:
# [EXPLORE] ──────────────────────────────────────────────────────────────────────────────
# Free workspace. Some ideas to explore:
#
# A. Apply the solver to Mars (R = 3.3895e6 m, rho = 3933 kg/m^3, g_ref = 3.72 m/s^2)
#    and compare the model prediction to the reference value.
#
# B. Make a single normalised plot of g(r)/g_surface vs. r/R for Earth, Jupiter,
#    and Mars. What does the normalised view reveal about the uniform-sphere model?
#
# C. Inspect the FD solution near r = R. Is the derivative of U_fd (i.e., g)
#    continuous there? Should it be? Compare your numerical gradient to analytical_gravity
#    just inside and just outside the surface.

# YOUR CODE HERE

---
## Synthesis

Answer the following questions in complete sentences. These questions draw across
all three Parts and require physical reasoning, not just computation.

### S1

Inside the sphere Poisson's equation holds ($\nabla^2 U = -4\pi G\rho_0$); outside,
Laplace's equation holds ($\nabla^2 U = 0$). Yet you solved both simultaneously on
a single grid without ever switching equations. How is this possible? What property
of your `density_profile` function is responsible?

**Your response:**

> *(Write 3–5 sentences.)*

### S2

Your VALIDATE cell printed the uniform-density surface gravity for both planets.
Fill in: $g_\text{Earth} \approx$ \_\_\_ m/s² (error: \_\_\_%) and
$g_\text{Jupiter} \approx$ \_\_\_ m/s² (error: \_\_\_%).

For each planet, name **one specific interior feature** that violates the constant-density
assumption and state whether it would increase or decrease $g_\text{surface}$ relative to
your computed value. Then address this: the same formula $g = \frac{4}{3}\pi G\rho_0 R$
applies to both planets — why might the uniform-density approximation be a *better or worse*
fit for one planet than the other?

**Your response:**

> *(Write 5–7 sentences. Fill in the blanks before answering.)*

### S3

Suppose you wanted to extend `solve_poisson_fd` to a **layered** planet — for example,
a dense metallic core surrounded by a lower-density mantle. Which part(s) of the
function would you need to modify? Which parts could remain unchanged? Sketch in words
(not code) what a two-layer `density_profile` would look like.

**Your response:**

> *(Write 4–6 sentences.)*

---
## Extensions *(optional)*

These are not graded unless stated by the instructor. They are for students who
finish early or want to go deeper.

### E1: Neumann vs. Dirichlet at the outer boundary

In `solve_poisson_fd` you applied a Dirichlet condition at $r = r_\text{max}$:
$u(r_\text{max}) = GM$. Alternatively, a Neumann (derivative) condition
$du/dr|_{r_\text{max}} = 0$ (equivalent to $dU/dr = -GM/r_\text{max}^2$) avoids
needing to know $M$ in advance. Modify the solver to use a Neumann outer BC and
compare the two solutions as $r_\text{max}/R$ decreases toward 1. Which converges
faster?

### E2: PREM-like layered density

Approximate Earth’s interior with a three-layer density profile:

| Layer | Radius range | Density |
|-------|--------------|---------|
| Core (inner + outer) | $0$ – $3.5 \times 10^6$ m | 11 000 kg/m$^3$ |
| Mantle | $3.5$ – $6.3 \times 10^6$ m | 4 500 kg/m$^3$ |
| Crust  | $6.3$ – $6.371 \times 10^6$ m | 2 800 kg/m$^3$ |

Modify `density_profile` for this structure, run the solver, and compare the surface
gravity and interior $g(r)$ profile to your uniform-density result. Where is the
difference largest?

### E3: Sparse solver

The current solver uses `np.linalg.solve`, which requires $O(N^3)$ work for a dense
matrix. The Poisson matrix is tridiagonal (only three non-zero diagonals), so a sparse
solver can handle it in $O(N)$ time. Replace `np.linalg.solve` with
`scipy.linalg.solve_banded` and verify that results match. How large an $N$ can you
now run in under one second?

In [ ]:
# [EXPLORE] ──────────────────────────────────────────────────────────────────────────────
# Extension workspace. Add cells as needed.

# YOUR CODE HERE